In [ ]:
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, ConfusionMatrixDisplay
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')


In [ ]:
df = pd.DataFrame([{'query': 'What are common symptoms of diabetes?', 'intent': 'symptoms'}, {'query': 'What signs indicate high blood sugar?', 'intent': 'symptoms'}, {'query': 'How can I tell if I have diabetes symptoms?', 'intent': 'symptoms'}, {'query': 'What symptoms occur in type 2 diabetes?', 'intent': 'symptoms'}, {'query': 'What are warning signs of diabetes?', 'intent': 'symptoms'}, {'query': 'Can diabetes cause frequent urination?', 'intent': 'symptoms'}, {'query': 'Why does diabetes make people thirsty?', 'intent': 'symptoms'}, {'query': 'What symptoms should I watch for with diabetes?', 'intent': 'symptoms'}, {'query': 'What foods are good for diabetes?', 'intent': 'diet'}, {'query': 'What should a diabetic eat for breakfast?', 'intent': 'diet'}, {'query': 'Which foods should I avoid with diabetes?', 'intent': 'diet'}, {'query': 'Can people with diabetes eat fruits?', 'intent': 'diet'}, {'query': 'What is a healthy diet for diabetes?', 'intent': 'diet'}, {'query': 'Are whole grains suitable for diabetes?', 'intent': 'diet'}, {'query': 'What drinks are better for diabetes?', 'intent': 'diet'}, {'query': 'How should I plan meals for diabetes?', 'intent': 'diet'}, {'query': 'What medicines are used to treat diabetes?', 'intent': 'medication'}, {'query': 'How does metformin help diabetes?', 'intent': 'medication'}, {'query': 'When are insulin injections used?', 'intent': 'medication'}, {'query': 'What is the purpose of diabetes medication?', 'intent': 'medication'}, {'query': 'Can diabetes medicine lower blood sugar?', 'intent': 'medication'}, {'query': 'What should I know about metformin?', 'intent': 'medication'}, {'query': 'How is insulin used in diabetes?', 'intent': 'medication'}, {'query': 'Are there different medicines for diabetes?', 'intent': 'medication'}, {'query': 'How often should blood sugar be checked?', 'intent': 'monitoring'}, {'query': 'What is a normal blood glucose reading?', 'intent': 'monitoring'}, {'query': 'How can I monitor my blood sugar?', 'intent': 'monitoring'}, {'query': 'What does HbA1c measure?', 'intent': 'monitoring'}, {'query': 'How often should a diabetic check glucose?', 'intent': 'monitoring'}, {'query': 'What does a glucose meter do?', 'intent': 'monitoring'}, {'query': 'Why is HbA1c important?', 'intent': 'monitoring'}, {'query': 'How can I keep track of blood sugar?', 'intent': 'monitoring'}, {'query': 'Is exercise helpful for diabetes?', 'intent': 'lifestyle'}, {'query': 'How much should a diabetic exercise?', 'intent': 'lifestyle'}, {'query': 'Can walking help control blood sugar?', 'intent': 'lifestyle'}, {'query': 'How does sleep affect diabetes?', 'intent': 'lifestyle'}, {'query': 'What lifestyle changes help diabetes?', 'intent': 'lifestyle'}, {'query': 'Can losing weight improve diabetes?', 'intent': 'lifestyle'}, {'query': 'Is regular physical activity important for diabetes?', 'intent': 'lifestyle'}, {'query': 'How can stress affect blood sugar?', 'intent': 'lifestyle'}, {'query': 'I am unconscious and have diabetes', 'intent': 'emergency'}, {'query': 'My blood sugar is extremely low and I feel confused', 'intent': 'emergency'}, {'query': 'I am having trouble breathing and feel very weak', 'intent': 'emergency'}, {'query': 'I think I am having a severe diabetic emergency', 'intent': 'emergency'}, {'query': 'My diabetic family member is unresponsive', 'intent': 'emergency'}, {'query': 'I have severe confusion with very low blood sugar', 'intent': 'emergency'}, {'query': 'There is a serious diabetes emergency happening', 'intent': 'emergency'}, {'query': 'I am fainting because of a diabetes problem', 'intent': 'emergency'}])
df.head()


In [ ]:
taxonomy = {'Clinical Information': ['Symptoms', 'Complications'], 'Nutrition': ['Food Choices', 'Meal Planning'], 'Treatment': ['Medication', 'Insulin'], 'Self Management': ['Monitoring', 'Lifestyle']}
for category, subcategories in taxonomy.items():
    print(category, '->', ', '.join(subcategories))


In [ ]:
concepts = ['Diabetes', 'Blood Glucose', 'Insulin', 'Metformin', 'HbA1c', 'Diet', 'Exercise', 'Symptoms', 'Medication', 'Emergency']
relationships = [('Diabetes', 'causes', 'Blood Glucose changes'), ('Insulin', 'regulates', 'Blood Glucose'), ('Metformin', 'helps control', 'Blood Glucose'), ('HbA1c', 'measures', 'long-term Blood Glucose'), ('Diet', 'affects', 'Blood Glucose'), ('Exercise', 'can improve', 'Blood Glucose control'), ('Diabetes', 'can cause', 'Symptoms'), ('Medication', 'treats', 'Diabetes'), ('Insulin', 'is a type of', 'Medication'), ('Severe Blood Glucose changes', 'can require', 'Emergency care')]
print('Domain Concepts:')
for concept in concepts:
    print('-', concept)
print('\nRelationships:')
for a, r, b in relationships:
    print(f'{a} -> {r} -> {b}')


In [ ]:
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def preprocess(text):
    text = text.lower()
    tokens = re.findall(r'[a-zA-Z]+', text)
    tokens = [token for token in tokens if token not in stop_words]
    tokens = [lemmatizer.lemmatize(token) for token in tokens]
    return ' '.join(tokens)

df['processed_query'] = df['query'].apply(preprocess)
df[['query', 'processed_query', 'intent']].head(10)


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    df['processed_query'], df['intent'], test_size=0.25, random_state=42, stratify=df['intent']
)

vectorizer = TfidfVectorizer()
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

classifier = LogisticRegression(max_iter=1000)
classifier.fit(X_train_tfidf, y_train)
y_pred = classifier.predict(X_test_tfidf)

print('Accuracy:', accuracy_score(y_test, y_pred))
print('Precision:', precision_score(y_test, y_pred, average='weighted', zero_division=0))
print('Recall:', recall_score(y_test, y_pred, average='weighted', zero_division=0))
print('F1-score:', f1_score(y_test, y_pred, average='weighted', zero_division=0))


In [ ]:
cm = confusion_matrix(y_test, y_pred, labels=classifier.classes_)
display_cm = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=classifier.classes_)
display_cm.plot(xticks_rotation=45)
plt.tight_layout()
plt.show()


In [ ]:
knowledge_base = {'symptoms': 'Common diabetes symptoms can include increased thirst, frequent urination, increased hunger, fatigue, blurred vision and unexplained weight changes.', 'diet': 'A diabetes-friendly diet generally emphasizes vegetables, whole grains, lean proteins, fiber-rich foods and appropriate portions while limiting excess added sugar and refined carbohydrates.', 'medication': 'Diabetes treatment may include medicines such as metformin and, for some people, insulin. Treatment should be selected and adjusted by a qualified healthcare professional.', 'monitoring': 'Blood glucose monitoring and HbA1c testing help track diabetes control. Target ranges vary by person, so healthcare professionals should provide individualized targets.', 'lifestyle': 'Regular physical activity, balanced nutrition, adequate sleep, stress management and maintaining a healthy weight can support diabetes management.', 'emergency': 'Severe confusion, unconsciousness, fainting, difficulty breathing or suspected severe blood sugar abnormalities can require urgent medical attention. Contact local emergency services immediately.'}
for intent, information in knowledge_base.items():
    print(f'{intent}: {information}\n')


In [ ]:
emergency_terms = ['unconscious', 'unresponsive', 'fainting', 'faint', 'difficulty breathing', 'trouble breathing', 'severe confusion', 'extremely low', 'emergency']
diagnosis_terms = ['diagnose', 'diagnosis', 'do i have diabetes', 'do i have']
medication_change_terms = ['change my medicine', 'change medication', 'increase my dose', 'decrease my dose']

def is_emergency(text):
    text = text.lower()
    return any(term in text for term in emergency_terms)

def is_unsupported(text):
    text = text.lower()
    supported_terms = ['diabetes', 'blood sugar', 'glucose', 'insulin', 'metformin', 'hba1c', 'diabetic', 'diabetes medicine', 'diabetes diet']
    return not any(term in text for term in supported_terms)

def assistant(query, threshold=0.45):
    if is_emergency(query):
        return knowledge_base['emergency']
    if is_unsupported(query):
        return 'This assistant supports diabetes-related healthcare information. Please ask a diabetes-related question.'

    processed = preprocess(query)
    features = vectorizer.transform([processed])
    probabilities = classifier.predict_proba(features)[0]
    index = np.argmax(probabilities)
    intent = classifier.classes_[index]
    confidence = probabilities[index]

    if confidence < threshold:
        return 'I am not confident enough to answer this query. Please rephrase it or consult a healthcare professional.'
    if any(term in query.lower() for term in diagnosis_terms):
        return 'I cannot diagnose a medical condition. Please consult a qualified healthcare professional for diagnosis.'
    if any(term in query.lower() for term in medication_change_terms):
        return 'I cannot recommend changing medication or dosage. Please consult your healthcare professional.'
    return knowledge_base[intent]


In [ ]:
test_queries = ['What are the common symptoms of diabetes?', 'What foods should I eat if I have diabetes?', 'How does metformin help diabetes?', 'How often should I check my blood sugar?', 'Can exercise help control diabetes?', 'What signs should I look for if my blood sugar is high?', 'What is the treatment for a broken bone?', 'Can you tell me about diabetes complications?', 'My diabetic father is unconscious and not responding.']
for query in test_queries:
    print('Query:', query)
    print('Response:', assistant(query))
    print('-' * 80)


In [ ]:
while True:
    query = input("Enter your healthcare query (type 'exit' to stop): ")
    if query.lower().strip() == 'exit':
        break
    print('\nAssistant:', assistant(query))
    print()
